In [1]:
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("DESeq2")

Bioconductor version '3.16' is out-of-date; the current release version '3.19'
  is available with R version '4.4'; see https://bioconductor.org/install

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.r-project.org

Bioconductor version 3.16 (BiocManager 1.30.20), R 4.2.2 (2022-10-31)

Warning message:
“package(s) not installed when version(s) same as or greater than current; use
  `force = TRUE` to re-install: 'DESeq2'”
Old packages: 'ape', 'askpass', 'bayesm', 'BH', 'BiocManager', 'boot', 'brew',
  'broom', 'bslib', 'callr', 'cli', 'clock', 'cluster', 'codetools',
  'commonmark', 'compositions', 'cpp11', 'crosstalk', 'curl', 'cyclocomp',
  'data.table', 'DBI', 'dbplyr', 'dbscan', 'deldir', 'DEoptimR', 'desc',
  'digest', 'dplyr', 'e1071', 'evaluate', 'fansi', 'FNN', 'fontawesome',
  'foreign', 'fs', 'future', 'future.apply', 'gargle', 'ggplot2', 'glmnet

In [121]:
library(DESeq2)
library(tidyverse)

In [138]:
counts_data <- read.csv("mrna.csv")
col_data <- read.csv("annotations.csv")

disease_data <- col_data
risk_data <- col_data[col_data$X2.risk != 0, ]
mutation_data <- col_data[col_data$X3.mutations..SF3B1only_wt. != 0, ]

disease_data$X1.disease <- factor(disease_data$X1.disease)
risk_data$X2.risk <- factor(risk_data$X2.risk)
mutation_data$X3.mutations..SF3B1only_wt. <- factor(mutation_data$X3.mutations..SF3B1only_wt.)

head(counts_data)
head(col_data)

,GENE_NAME,V1884,N58,V630,N60,V1297,NV1428,N82,V940,V2092,⋯,V67,V1090,V1860,V406,V1834,V1048,V806,V513,V1565,V1920
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,RILPL1,187,235,538,244,256,131,89,364,362,⋯,432,207,373,180,301,404,458,263,384,649
2,RAB4B,1296,951,1261,856,613,849,749,813,937,⋯,1374,812,1238,1024,1072,764,801,989,1682,943
3,TIGAR,1229,184,609,691,173,325,393,227,424,⋯,408,240,340,564,371,404,499,400,508,204
4,DNAH3,5,0,12,16,12,3,30,2,92,⋯,6,0,7,3,115,0,3,0,11,22
5,RP11-432M8.3,2,1,1,0,3,0,2,1,7,⋯,1,0,3,1,11,0,0,0,0,3
6,RPL23A,27871,36642,41933,28742,15317,39334,29076,37869,24541,⋯,56969,44306,30206,32677,19284,29511,36101,24800,80018,23720


,SAMPLE_NAME,X1.disease,X2.risk,X3.mutations..SF3B1only_wt.,sample_ids
,<chr>,<int>,<int>,<int>,<chr>
1,V1884_S17,2,2,0,V1884
2,N58_S18,1,0,0,N58
3,V630_S11,2,1,1,V630
4,N60_S15,1,0,0,N60
5,V1297_S10,2,2,0,V1297
6,NV1428_S3,1,0,0,NV1428


In [139]:
counts_data <- counts_data[!duplicated(counts_data$GENE_NAME), ]
rownames(counts_data) <- counts_data$GENE_NAME
counts_data$GENE_NAME <- NULL
counts_data_filt <- counts_data[, colnames(counts_data) %in% mutation_data$sample_ids]

In [140]:
dds <- DESeqDataSetFromMatrix(countData = counts_data_filt, colData = mutation_data, design = ~ X3.mutations..SF3B1only_wt.)

In [141]:
dds

class: DESeqDataSet 
dim: 19871 26 
metadata(1): version
assays(1): counts
rownames(19871): RILPL1 RAB4B ... PPP6R1 OR8D4
rowData names(0):
colnames(26): V630 V624 ... V806 V513
colData names(5): SAMPLE_NAME X1.disease X2.risk
  X3.mutations..SF3B1only_wt. sample_ids

In [142]:
keep <- rowSums(counts(dds)) >= 50
dds <- dds[keep, ]

In [68]:
dds$X1.disease <- relevel(dds$X1.disease, ref="1")

In [127]:
dds$X2.risk <- relevel(dds$X2.risk, ref="1")

In [143]:
dds$X3.mutations..SF3B1only_wt. <- relevel(dds$X3.mutations..SF3B1only_wt., ref="1")

In [116]:
dds$X2.risk

[1] 2 1 2 1 1 2 2 1 1 2 1 1 1 2 1 2 1 2 2 1 1 1 1 2 1 2 1 2 1 2 1 1 1 1 2 1 2 1
[39] 2 2 1 2 2 2 1 2 1 1 1 2 2 2 1
Levels: 1 2

In [144]:
dds <- DESeq(dds)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

-- replacing outliers and refitting for 3398 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)

estimating dispersions

fitting model and testing



In [145]:
res <- results(dds)

In [146]:
res

log2 fold change (MLE): X3.mutations..SF3B1only wt. 2 vs 1 
Wald test p-value: X3.mutations..SF3B1only wt. 2 vs 1 
DataFrame with 18696 rows and 6 columns
          baseMean log2FoldChange     lfcSE      stat    pvalue      padj
         <numeric>      <numeric> <numeric> <numeric> <numeric> <numeric>
RILPL1   257.32673      0.0220592  0.331234  0.066597  0.946903  0.994156
RAB4B   1086.43297     -0.1291706  0.150854 -0.856261  0.391853  0.909896
TIGAR    396.35600      0.0996125  0.231044  0.431140  0.666366  0.964759
DNAH3      6.03072      0.2242165  0.737335  0.304090  0.761059        NA
RPL23A 33318.34020      0.0774624  0.130695  0.592698  0.553384  0.947555
...            ...            ...       ...       ...       ...       ...
CYP4F2   144.18520     -1.1177145  0.580047 -1.926936 0.0539876  0.535543
TENM1     69.75578      0.1958124  0.660460  0.296479 0.7668645  0.979940
BATF3    140.41595     -0.0808827  0.384455 -0.210382 0.8333693  0.985931
PPP6R1 21099.34451      0.18864

In [147]:
res_ordered <- res[order(abs(res$stat), decreasing = TRUE), ]
write.table(res_ordered[1:30, ], file = "mutationbest30.csv", row.names = TRUE, sep = "\t", quote = FALSE)
# res_ordered <- res[order(abs(res$stat, , decreasing = TRUE)), ]

In [89]:
head(res_ordered, n=30)

log2 fold change (MLE): X1.disease 2 vs 1 
Wald test p-value: X1.disease 2 vs 1 
DataFrame with 30 rows and 6 columns
                baseMean log2FoldChange     lfcSE      stat      pvalue
               <numeric>      <numeric> <numeric> <numeric>   <numeric>
C9orf129          57.095       22.12170  1.458149  15.17108 5.49611e-52
CTD-2643I7.4    1296.129        5.50993  0.661358   8.33123 8.00038e-17
HBG1            1263.233        5.58540  0.696632   8.01772 1.07724e-15
HBG2            1163.078        5.34515  0.670219   7.97523 1.52102e-15
RELL1           2197.964       -1.39906  0.181634  -7.70261 1.33314e-14
...                  ...            ...       ...       ...         ...
ADGRB1          464.0348        2.41173  0.395753   6.09401 1.10116e-09
HBB          103376.7246        3.44195  0.565833   6.08298 1.17971e-09
STXBP5L         167.5345        3.38409  0.569238   5.94495 2.76543e-09
PRAME            93.3052        4.11247  0.693791   5.92754 3.07513e-09
TSPO2           16

In [104]:
options(dplyr.print_max = 1e9)

In [101]:
getOption("max.print")

[1] 1000

In [105]:
head(res_ordered, n=30)

log2 fold change (MLE): X1.disease 2 vs 1 
Wald test p-value: X1.disease 2 vs 1 
DataFrame with 30 rows and 6 columns
                baseMean log2FoldChange     lfcSE      stat      pvalue
               <numeric>      <numeric> <numeric> <numeric>   <numeric>
C9orf129          57.095       22.12170  1.458149  15.17108 5.49611e-52
CTD-2643I7.4    1296.129        5.50993  0.661358   8.33123 8.00038e-17
HBG1            1263.233        5.58540  0.696632   8.01772 1.07724e-15
HBG2            1163.078        5.34515  0.670219   7.97523 1.52102e-15
RELL1           2197.964       -1.39906  0.181634  -7.70261 1.33314e-14
...                  ...            ...       ...       ...         ...
ADGRB1          464.0348        2.41173  0.395753   6.09401 1.10116e-09
HBB          103376.7246        3.44195  0.565833   6.08298 1.17971e-09
STXBP5L         167.5345        3.38409  0.569238   5.94495 2.76543e-09
PRAME            93.3052        4.11247  0.693791   5.92754 3.07513e-09
TSPO2           16